# BAH 2026: RIFE HDv3 Satellite Fine-Tuning

This notebook is designed to run on Google Colab (Free Tier) to fine-tune the RIFE HDv3 model on geostationary satellite thermal infrared (TIR) imagery.

**Objectives:**
1. Setup environment and download pre-trained RIFE weights.
2. Download real GOES-19 NetCDF data from AWS S3.
3. Process into triplets (T-1, T, T+1) for optical flow training.
4. Fine-tune the model to capture non-linear cloud dynamics.

In [ ]:
!pip install torch torchvision numpy opencv-python tqdm xarray netCDF4 h5py s3fs

## 1. Download Dataset

In [ ]:
# Assuming prepare_goes_data.py is in the same directory (or cloned from your repo)
# This will download ~30 NetCDF files and create image triplets
!python prepare_goes_data.py --satellite goes19 --max_files 30 --out_dir ./goes_dataset

## 2. Training Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

# Import our custom dataset
from satellite_dataset import SatelliteVimeoDataset

# Import model (Ensure IFNet_HDv3.py and RIFE_HDv3.py are accessible)
import sys
sys.path.append('../FastAPI/train_log')
from RIFE_HDv3 import Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
dataset = SatelliteVimeoDataset('./goes_dataset/triplets', is_training=True, crop_size=(256, 256))
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)

print(f"Loaded {len(dataset)} training triplets.")

In [ ]:
# Initialize Model
model = Model()
model.device()

# Load pre-trained weights to start from a good baseline
# model.load_model('../FastAPI/train_log', -1)

model.flownet.train()

# Optimizer with low learning rate for fine-tuning
optimizer = optim.AdamW(model.flownet.parameters(), lr=1e-5, weight_decay=1e-4)
criterion = nn.MSELoss()

## 3. Fine-Tuning Loop

In [ ]:
epochs = 5

for epoch in range(epochs):
    epoch_loss = 0.0
    
    progress = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
    for img0, gt, img1 in progress:
        img0 = img0.to(device)
        gt = gt.to(device)
        img1 = img1.to(device)
        
        optimizer.zero_grad()
        
        # RIFE uses recursive inference, but for training a single step:
        # We'd typically use model.update() or forward pass of flownet directly.
        # Assuming standard forward pass for simplification in this snippet:
        pred = model.inference(img0, img1)
        
        loss = criterion(pred, gt)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        progress.set_postfix({'loss': loss.item()})
        
    print(f"Epoch {epoch+1} Average Loss: {epoch_loss / len(dataloader):.6f}")

    # Save checkpoint
    model.save_model('./finetuned_models', epoch)